# PySpark 3: joins and aggregations

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-03-joins-and-aggregations.ipynb)

Joins and aggregations decide how much data moves across the cluster. This module shows the join types you will use, when to broadcast a small table, and how groupBy and conditional aggregates summarize data.

**You will learn to:**

- Choose between a shuffle join and a broadcast join
- Write left, anti and semi joins for real questions
- Summarize with `groupBy` and conditional aggregates
- Recognise where a shuffle is the cost

**Before you start:** PySpark 1 and 2.

## Setup and sample data

This module runs on the same four tables as PySpark 1, so each module can run on its own. The setup cell installs PySpark when it is missing and starts a local session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter, and every timestamp in the lessons is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-lessons-")

spark = (
    SparkSession.builder
    .appName("pyspark-lessons")
    .master("local[2]")  # driver and executor share one JVM: the development topology
    .config("spark.sql.shuffle.partitions", "4")  # 200 is the default; tiny data needs few
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## Sample data: the tables every lesson below runs on

This section seeds a small four-table slice of an orders platform, built with
`spark.createDataFrame` and an explicit schema per table. Three quirks the lessons lean on:
customer 4 has no `region`, customers 7 and 8 have never ordered, and one event is an exact
retry of the same row.

| table | grain | notable columns |
| --- | --- | --- |
| `customers` | one row per customer | `region`, `signup_date` (`DATE`) |
| `orders` | one row per order | `status`, `order_ts` (`TIMESTAMP`), `amount` (decimal) |
| `order_items` | one row per order line | `qty`, `unit_price` |
| `events` | one row per user event | `user_id`, `event`, `ts` |

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import types as T

customers_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("signup_date", T.DateType(), False),
])

customers = spark.createDataFrame(
    [
        (1, "Acme Corp",      "North", date(2024, 1, 3)),
        (2, "Bright Retail",  "North", date(2024, 2, 11)),
        (3, "Cyan Traders",   "South", date(2024, 3, 15)),
        (4, "Delta Grocers",  None,    date(2024, 5, 2)),
        (5, "Echo Supplies",  "West",  date(2024, 6, 20)),
        (6, "Foxtrot Foods",  "West",  date(2024, 8, 1)),
        (7, "Gamma Sports",   "East",  date(2024, 9, 14)),
        (8, "Hotel Homeware", "East",  date(2024, 11, 5)),
    ],
    schema=customers_schema,
)

customers.orderBy("customer_id").show()

In [ ]:
from pyspark.sql import types as T

orders_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("status", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), False),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders = spark.createDataFrame(
    [
        (1,  1, "paid",     datetime(2026, 1,  5,  9, 15), D("120.00")),
        (2,  1, "shipped",  datetime(2026, 1,  5, 10,  5),  D("84.50")),
        (3,  1, "returned", datetime(2026, 1,  5, 11, 40),  D("30.00")),
        (4,  2, "paid",     datetime(2026, 1,  6, 15, 20), D("260.00")),
        (5,  2, "placed",   datetime(2026, 1,  7,  8,  5),  D("55.25")),
        (6,  3, "shipped",  datetime(2026, 1,  7, 17, 45), D("149.00")),
        (7,  3, "paid",     datetime(2026, 1,  9, 12, 10), D("310.00")),
        (8,  4, "returned", datetime(2026, 1, 10, 19, 30),  D("89.90")),
        (9,  4, "shipped",  datetime(2026, 1, 12,  8, 55), D("1180.00")),
        (10, 5, "paid",     datetime(2026, 1, 12, 13,  0),   D("74.00")),
        (11, 6, "placed",   datetime(2026, 1, 13, 10, 25),  D("210.00")),
        (12, 6, "shipped",  datetime(2026, 1, 14,  0, 30),   D("96.40")),
    ],
    schema=orders_schema,
)

orders.orderBy("order_id").show()
orders.printSchema()

In [ ]:
from pyspark.sql import types as T

order_items_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("sku", T.StringType(), False),
    T.StructField("qty", T.IntegerType(), False),
    T.StructField("unit_price", T.DecimalType(8, 2), False),
])

order_items = spark.createDataFrame(
    [
        (1,  "SKU-4001",  2,  D("12.50")),
        (1,  "SKU-4007",  1,  D("95.00")),
        (2,  "SKU-4002",  1,  D("34.50")),
        (2,  "SKU-4009",  2,  D("25.00")),
        (3,  "SKU-4002",  1,  D("30.00")),
        (4,  "SKU-4003",  1, D("149.00")),
        (4,  "SKU-4004",  2,  D("55.50")),
        (5,  "SKU-4005",  1,  D("55.25")),
        (6,  "SKU-4003",  1, D("149.00")),
        (7,  "SKU-4006",  1, D("189.00")),
        (7,  "SKU-4008",  1, D("121.00")),
        (8,  "SKU-4010",  1,  D("89.90")),
        (9,  "SKU-4011", 20,  D("59.00")),
        (10, "SKU-4012",  2,  D("37.00")),
        (11, "SKU-4013",  1, D("210.00")),
        (12, "SKU-4014",  1,  D("41.40")),
        (12, "SKU-4005",  1,  D("55.00")),
    ],
    schema=order_items_schema,
)

order_items.orderBy("order_id", "sku").show(20)

In [ ]:
from pyspark.sql import types as T

events_schema = T.StructType([
    T.StructField("user_id", T.IntegerType(), False),
    T.StructField("event", T.StringType(), False),
    T.StructField("ts", T.TimestampType(), False),
])

events = spark.createDataFrame(
    [
        (1, "view",        datetime(2026, 1,  6,  9,  0)),
        (1, "add_to_cart", datetime(2026, 1,  6,  9,  2)),
        (1, "purchase",    datetime(2026, 1,  6,  9, 10)),
        (2, "view",        datetime(2026, 1,  7, 10,  0)),
        (2, "purchase",    datetime(2026, 1,  7, 10, 40)),
        (3, "view",        datetime(2026, 1,  6, 11, 15)),
        (3, "view",        datetime(2026, 1,  8, 11, 20)),
        (4, "add_to_cart", datetime(2026, 1,  9, 20,  5)),
        (4, "purchase",    datetime(2026, 1, 10,  8, 30)),
        (5, "view",        datetime(2026, 1, 11, 14, 45)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
        (5, "purchase",    datetime(2026, 1, 12, 12, 30)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
    ],
    schema=events_schema,
)

events.orderBy("user_id", "ts").show()

In [ ]:
from pyspark.sql import functions as F

frames = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "events": events,
}
expected_rows = {"customers": 8, "orders": 12, "order_items": 17, "events": 13}

for table, frame in frames.items():
    actual = frame.count()
    assert actual == expected_rows[table], (table, actual, expected_rows[table])
    print(f"{table:<12} {actual:>2} rows")

per_order = (
    order_items
    .groupBy("order_id")
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("lines"))
    .join(orders.select("order_id", "amount"), "order_id")
)
assert per_order.where(F.col("lines") != F.col("amount")).count() == 0
print("line totals reconcile with the orders ledger:", float(orders.agg(F.sum("amount")).first()[0]))

assert events.dropDuplicates().count() == events.count() - 1
print("and the retried event collapses to 12 distinct rows - seed verified")

In [ ]:
import json
from pyspark.sql import Window, functions as F
from pyspark.sql.functions import broadcast

In [ ]:
daily_revenue = (
    orders
    .where(F.col("status").isin("paid", "shipped"))
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)
print(type(daily_revenue).__name__, "- nothing has executed yet")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()


orders_path = os.path.join(WORK_DIR, "orders_parquet")
orders.write.mode("overwrite").parquet(orders_path)

result = (
    spark.read.parquet(orders_path)
    .where("order_ts >= timestamp '2026-01-10'")
    .select("order_id", "customer_id", "amount")
    .where("amount > 0")
)
plan = plan_of(result)
print(plan)
read_schema = next(line for line in plan.splitlines() if line.startswith("ReadSchema"))
assert "GreaterThan(amount,0.00)" in plan  # the filter reached the Parquet reader
assert "status" not in read_schema  # unused columns are never read

You can now predict what will and will not execute. This module is where frames meet data:
joins, aggregations and window functions are the wide operations that move bytes across
the cluster, and the cookbook closes it as the everyday recipe list. Every lesson below
reads the sample-data tables from PySpark 1, so the outputs shown are the outputs a reader
runs.

## How do you join data in PySpark?

A join combines DataFrames by key. The most important production questions are key uniqueness, null handling, data skew, and whether Spark should shuffle both sides or broadcast the small side.

In [ ]:
from pyspark.sql import functions as F

enriched = (
    orders.alias("o")
    .join(customers.alias("c"), on="customer_id", how="left")
    .select(
        "o.order_id",
        "o.customer_id",
        "c.name",
        "c.region",
        "o.status",
        "o.amount",
    )
    .orderBy("o.order_id")
)
enriched.show()

# `customer_id` is unique on the dimension, so the enrich join must not fan out
assert enriched.count() == orders.count()
print("join verified: one row per order")

If the dimension is small enough, broadcast it:

In [ ]:
from pyspark.sql.functions import broadcast

# eight tiny rows on the build side: every executor should hold `customers`,
# and nothing about `orders` needs to move at all
enriched = orders.join(broadcast(customers), "customer_id", "left")

assert enriched.count() == orders.count()

**Where it matters.** broadcasting a 20 MB customer segment dimension can avoid a large shuffle when joining it to a 3 TB orders fact table. Do not broadcast a dimension just because it is called a dimension; verify its size.

**How it works.** A sort-merge join shuffles both sides by the key hash, sorts each partition, then merges. A broadcast hash join ships the small side to every executor, builds a hash table there, and streams the large side past it with no shuffle at all. Spark picks broadcast when the estimated side size is under `spark.sql.autoBroadcastJoinThreshold`, and a `broadcast()` hint overrides the estimate.

In [ ]:
from pyspark.sql.functions import broadcast

small_customers = customers.select("customer_id", "name", "region")

enriched = (
    orders.alias("o")
    .join(broadcast(small_customers).alias("c"), on="customer_id", how="left")
    .select("o.order_id", "o.amount", "o.status", "c.name", "c.region")
)
plan = plan_of(enriched)
print(plan)
assert "BroadcastHashJoin" in plan and "SortMergeJoin" not in plan

**Example.** A 3 TB orders fact joined to a 20 MB customer dimension: sort-merge shuffles about 3 TB, while broadcast sends 20 MB to each executor and shuffles nothing. Across 100 executors that is about 2 GB of network instead of terabytes.

**Watch out for.** broadcasting a "dimension" that is actually 2 GB. Every executor must hold the whole build side, so you get OOM or a silently ignored hint; confirm `BroadcastHashJoin` in the plan.

## How do you aggregate data in PySpark?

Aggregations summarize records by one or more grouping columns. They usually create a shuffle because rows with the same key must be brought together.

In [ ]:
from pyspark.sql import functions as F

sku_sales = (
    order_items
    .withColumn("line_amount", F.col("qty") * F.col("unit_price"))
    .groupBy("sku")
    .agg(
        F.countDistinct("order_id").alias("orders"),
        F.sum("qty").alias("units"),
        F.sum("line_amount").alias("revenue"),
    )
    .orderBy("sku")
)
sku_sales.show(20)

# SKU-4005 is carried by orders 5 and 12: two orders, two units, 55.25 + 55.00
row = sku_sales.where("sku = 'SKU-4005'").first()
assert row["orders"] == 2 and row["units"] == 2
assert round(float(row["revenue"]), 2) == 110.25

For conditional metrics, use expressions:

In [ ]:
from pyspark.sql import functions as F

customer_metrics = (
    orders
    .groupBy("customer_id")
    .agg(
        F.sum("amount").alias("lifetime_value"),
        F.sum(F.when(F.col("status") == "returned", F.col("amount")).otherwise(0)).alias("returned_amount"),
        F.max("order_ts").alias("last_order_ts"),
    )
    .orderBy("customer_id")
)
customer_metrics.show()

# customer 1: 120.00 + 84.50 + the fully returned 30.00 order
row = customer_metrics.where("customer_id = 1").first()
assert row["lifetime_value"] == 234.50 and row["returned_amount"] == 30.00
print("customers 7 and 8 are missing from the result - they have no orders to group")

Interview point: aggregations often need tuning because they shuffle data. Check the Spark UI for skewed shuffle partitions, spilled memory, and long-running reduce tasks.

**How it works.** A hash aggregate first runs a partial aggregate inside each map task, exchanges the reduced rows by group key, then runs the final aggregate. That partial step is why `sum` and `count` shuffle so much less than `countDistinct`, which must ship the distinct values themselves because they cannot be combined from partial sets exactly.

**Example.** 1 billion rows at 100 bytes each is 100 GB; a partial aggregate over 50 million keys can cut that to roughly 200 million intermediate rows, so the shuffle carries about 20 GB instead of 100 GB.

**Watch out for.** `countDistinct` on a high-cardinality column inside a big `groupBy`. Use `approx_count_distinct` when a small error is acceptable, or deduplicate first and count rows.

## Validate join cardinality before aggregating

A table's **grain** is what one row represents. If the orders table has one row per order, joining it to a customer dimension with two rows for the same key produces two rows for every matching order. Summing order amounts after that join counts the amount twice.

**Where it matters.** Check uniqueness of dimension keys before enriching a fact table. After a left join to a unique dimension, compare output and input row counts and measure unmatched keys separately.

**Watch out for.** A total row-count check alone can miss errors in an inner join because missing matches can cancel out extra matches. Fix the duplicate dimension key with a business rule, such as selecting the version valid at the order time.

### Example

Join one order worth 2500 cents to two customer records. The uniqueness check finds the bad key, and the final assertion shows what would happen to revenue if the join went ahead.

In [ ]:
orders_at_order_grain = spark.createDataFrame([(101, 7, 2500)], "order_id long, customer_id long, amount_cents long")
customer_versions = spark.createDataFrame([(7, "east"), (7, "west")], "customer_id long, region string")
duplicate_keys = customer_versions.groupBy("customer_id").count().where("count > 1")
assert [row.customer_id for row in duplicate_keys.collect()] == [7]

multiplied_orders = orders_at_order_grain.join(customer_versions, "customer_id", "left")
assert multiplied_orders.agg(F.sum("amount_cents")).first()[0] == 5000

## Check yourself

The lessons above end with takeaways. Re-run each cell in order, and use `explain` on any query you do not understand before you move on.

## Key takeaways

- Broadcast the small side of a join. A 20 MB dimension should never shuffle a 3 TB fact.
- Aggregations shuffle. Partial aggregation before the shuffle keeps that cost down.
- Use an anti join to find missing keys; a left join with a null check gets the same answer more slowly.

**Next:** [PySpark 4: windows and nested data](pyspark-04-windows-and-nested-data.ipynb)